# PLE Offline Engineering Lab (CPU-only, no Docker, no network, no model API)

Runs the harness offline test suite, validates the deterministic engineering solvers against hand-checked values, exercises the mock protocol clients, analyzes the committed baseline artifacts, and produces the analysis figures and a Stage-1 writeup draft with explicit evidence gaps.

**This is offline engineering and verification — NOT an official benchmark evaluation.** Figures and text state their provenance; nothing here presents synthetic numbers as competition results.

Setup: upload the repository as a Kaggle dataset and point `REPO` at its path (the default assumes the notebook sits inside the repo). Internet: OFF. Accelerator: None.

In [ ]:
import pathlib, sys
REPO = pathlib.Path.cwd()
while not (REPO / 'kaggle' / 'ple_offline_lab.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('repo root:', REPO)

## 1. Offline test suite (pass/fail matrix by module)

In [ ]:
from kaggle.ple_offline_lab import run_test_suite
SUITE = run_test_suite()
assert SUITE['failures'] == 0 and SUITE['errors'] == 0, 'offline suite must pass for later steps to be meaningful'
SUITE

## 2. Deterministic solver validation (hand-checked values)

In [ ]:
from kaggle.ple_offline_lab import validate_solvers
SOLVERS = validate_solvers()
SOLVERS

## 3. Failure-injection checks (the engineering failure cases live in the suite)

In [ ]:
import unittest
from kaggle.ple_offline_lab import REPO_ROOT
loader = unittest.TestLoader()
suite = unittest.TestSuite()
# the required failure cases: superseded documents, preconsolidation boundary,
# percent/fraction errors, NaN fields, missing register rows, wrong paths,
# malformed verdicts, verifier tampering, protocol cancellation
for module, cases in {
    'tests.test_authority': ['ResolutionTests'],
    'tests.test_contract': ['CsvCheckTests', 'ValidateDeliverablesTests'],
    'tests.test_solvers': ['CivilConsolidationTests', 'UnitsTests', 'ElectricalTests'],
    'tests.test_orchestrator': ['VerdictRuleTests', 'ParseJsonBlockTests'],
}.items():
    mod = __import__(module, fromlist=cases)
    for case in cases:
        suite.addTests(loader.loadTestsFromTestCase(getattr(mod, case)))
runner = unittest.TextTestRunner(verbosity=1)
result = runner.run(suite)
print(f"failure-injection cases: {result.testsRun - len(result.failures) - len(result.errors)}/{result.testsRun} passed")
assert result.wasSuccessful()

## 4. Mock protocol / MCP client exercise (loopback only)

In [ ]:
from kaggle.ple_offline_lab import exercise_mock_protocol
exercise_mock_protocol()

## 5. Baseline analysis (from the committed artifacts manifest — immutable evidence)

In [ ]:
from kaggle.ple_offline_lab import analyze_baseline
import pandas as pd
BASELINE = analyze_baseline()
if BASELINE:
    df = pd.DataFrame(sorted(BASELINE['rewards'].items(), key=lambda kv: kv[1]),
                      columns=['task', 'reward'])
    display(df)
    display(df.describe().T)

## 6. Figures (provenance printed with each)

`score_vs_cost.png` is a deliberate pending-data placeholder: the baseline trajectories carry no usage telemetry, and inventing costs is forbidden.

In [ ]:
from kaggle.ple_offline_lab import figure_architecture, figure_dev_failure_breakdown, figure_score_vs_cost
figure_architecture()
figure_dev_failure_breakdown(BASELINE)
figure_score_vs_cost()

In [ ]:
from IPython.display import Image, display
figs = pathlib.Path(REPO) / 'kaggle' / 'figures'
for name in ('architecture.png', 'dev_failure_breakdown.png', 'score_vs_cost.png'):
    p = figs / name
    if p.exists():
        print(name)
        display(Image(str(p), width=640))

## 7. Stage-1 writeup draft (placeholders preserved on purpose)

In [ ]:
from kaggle.ple_offline_lab import write_writeup
path = write_writeup()
print(path)
print(path.read_text()[:2000])

## Data handling

No raw competition task files, hidden-test material, or full trajectories belong in a public notebook or dataset. This lab reads only the committed `artifacts/manifest.json` (aggregate rewards) and synthetic fixtures, and the writeup uses placeholders wherever evidence is missing. See `kaggle/README.md`.